# Greffier, le modele : 01. Exploration des donnees (M-POPP)

Premiere brique du projet. Objectif : **regarder les vraies donnees avant tout entrainement**,
pour comprendre a quoi ressemblent les actes et comment les entites sont annotees.

Principe d'honnetete tenu tout le long du projet :
- jeux train / validation / test **separes par page** (jamais de fuite),
- tous les scores annonces viendront du **jeu de test**, jamais vu a l'entrainement,
- on montre aussi les echecs, on ne choisit pas les beaux exemples.

Dataset : M-POPP (projet EXO-POPP, INSA Rouen), actes de mariage parisiens 1880-1940,
licence CC-BY 4.0. https://zenodo.org/records/10980636


In [ ]:
# Environnement (Colab a deja la plupart de ces paquets)
import os, glob, json, random, collections, textwrap
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

random.seed(42)  # reproductibilite
RACINE = Path('m-popp')
RACINE.mkdir(exist_ok=True)


## 1. Telechargement
L'archive fait environ 1 Go. On ne la retelecharge pas si elle est deja la.


In [ ]:
ZIP = RACINE / 'm-popp_datasets.zip'
URL = 'https://zenodo.org/records/10980636/files/m-popp_datasets.zip?download=1'
if not ZIP.exists():
    print('Telechargement...')
    os.system(f'wget -q --show-progress -O "{ZIP}" "{URL}"')
else:
    print('Deja telecharge :', ZIP)

# Decompression
if not (RACINE / 'extrait').exists():
    os.system(f'unzip -q -o "{ZIP}" -d "{RACINE / "extrait"}"')
print('Contenu decompresse dans', RACINE / 'extrait')


## 2. Structure des fichiers
On explore l'arborescence pour voir comment c'est organise (manuscrit / imprime, splits, labels).


In [ ]:
base = RACINE / 'extrait'
for chemin in sorted(base.rglob('*')):
    prof = len(chemin.relative_to(base).parts)
    if prof <= 3 and chemin.is_dir():
        print('  '*(prof-1) + '[' + chemin.name + ']')

print('\n--- Comptage par extension ---')
ext = collections.Counter(p.suffix.lower() for p in base.rglob('*') if p.is_file())
for e, n in ext.most_common():
    print(f'{e or "(sans)":8} {n}')


## 3. Un exemple : image + annotation brute
On prend une page manuscrite d'entrainement au hasard, on l'affiche, et on lit son fichier
d'annotation **tel quel** pour decouvrir le format des balises d'entites.


In [ ]:
# Recherche defensive : on trouve les images, puis le label du meme nom
images = [p for p in base.rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png')
          and 'hand' in str(p).lower() and 'train' in str(p).lower()]
print('Images manuscrites (train) trouvees :', len(images))

def trouve_label(img):
    # cherche un fichier de meme radical avec une extension texte
    for ext in ('.txt','.json','.xml','.label'):
        for cand in [img.with_suffix(ext),
                     img.parent.parent / 'labels' / (img.stem + ext),
                     img.parent / 'labels' / (img.stem + ext)]:
            if cand.exists():
                return cand
    # sinon, on cherche par radical dans tout le dossier manuscrit
    for cand in base.rglob(img.stem + '.*'):
        if cand.suffix.lower() in ('.txt','.json','.xml','.label'):
            return cand
    return None

if images:
    img = random.choice(images)
    lab = trouve_label(img)
    print('Image :', img)
    print('Label :', lab)
    plt.figure(figsize=(9,12)); plt.imshow(Image.open(img)); plt.axis('off'); plt.show()
    if lab:
        contenu = lab.read_text(encoding='utf-8', errors='replace')
        print('\n--- Annotation brute (2000 premiers caracteres) ---\n')
        print(contenu[:2000])


## 4. Format des entites
M-POPP encode les entites **dans le texte**, avec des balises (5 encodages differents existent).
Ici on repere les balises presentes et on compte les types d'entites, pour savoir vers quel
jeu commun on harmonisera plus tard (avec Esposalles).


In [ ]:
import re
if images and lab:
    balises = re.findall(r'<[^>]+>', contenu)
    types = collections.Counter(balises)
    print('Types de balises dans cet acte :')
    for t, n in types.most_common():
        print(f'{n:3}  {t}')


## 5. Statistiques par split
Un comptage simple des pages par sous-ensemble, pour verifier qu'on a bien train / val / test
et pour garder en tete la **petite taille** (donc prudence sur les scores, et interet du synthetique).


In [ ]:
for jeu in ('train','val','valid','validation','test'):
    imgs = [p for p in base.rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png')
            and 'hand' in str(p).lower() and jeu in str(p).lower()]
    if imgs:
        print(f'manuscrit / {jeu:12} : {len(imgs)} images')


## Prochaines etapes
1. Faire de meme avec **Esposalles** (Espagne) et **harmoniser les etiquettes** vers un jeu commun.
2. Choisir le petit modele vision-langage a affiner (Florence-2 ou Qwen2-VL-2B).
3. Construire le **generateur d'actes synthetiques** (FR + ES) pour entrainer a grande echelle.
4. Pre-entrainement synthetique, puis fine-tuning sur le reel.
5. Comparaison honnete contre un gros generaliste (Gemini), meme jeu de test.

Rien ici n'entraine encore : on a juste **regarde les donnees en vrai**, comme il se doit.
